In [1]:
import pandas as pd
import folium
import branca.colormap as bcm

In [4]:
results = pd.read_csv('../results/geostatistics/spatial_interpolation_k159f4_results_per_station.csv',
                      dtype={'station_id': str})

coords = pd.read_csv('../data/wind_parameter.csv', sep=';',
                     dtype={'park_id': str})

results.head()

,station_id,method,rmse,mae,r2
0,00096,idw,1.467383,1.306567,-0.211585
1,00096,ok,1.828040,1.614073,-0.880350
2,00096,rk,1.454069,1.235549,-0.189700
3,00161,idw,1.586423,1.309459,-0.215646
4,00161,ok,1.812768,1.485187,-0.587279


In [5]:
# Methode wählen: 'idw', 'ok', 'rk', 'ws_from_uv_ok', ...
METHOD = 'rk'

df = results[results['method'] == METHOD].copy()
df = df.merge(coords[['park_id', 'latitude', 'longitude', 'altitude']],
              left_on='station_id', right_on='park_id', how='left')

print(f"Stationen: {len(df)}")
print(f"R² — min: {df['r2'].min():.3f}, median: {df['r2'].median():.3f}, mean: {df['r2'].mean():.3f}, max: {df['r2'].max():.3f}")

Stationen: 160
R² — min: -1.971, median: 0.718, mean: 0.616, max: 0.879


In [6]:
colormap = bcm.LinearColormap(
    colors=['#d73027', '#fee08b', '#1a9850'],  # rot → gelb → grün
    vmin=0, vmax=1,
    caption=f'R² ({METHOD.upper()})'
)

m = folium.Map(location=[51.5, 10.5], zoom_start=6, tiles='CartoDB positron')

for _, row in df.iterrows():
    r2 = row['r2']
    color = colormap(max(0, min(1, r2)))  # clamp [0,1]

    popup_html = (
        f"<b>Station {row['station_id']}</b><br>"
        f"R² = {r2:.3f}<br>"
        f"RMSE = {row['rmse']:.3f}<br>"
        f"MAE = {row['mae']:.3f}<br>"
        f"Höhe = {row['altitude']:.0f} m"
    )

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7,
        color='black',
        weight=0.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=200),
        tooltip=f"{row['station_id']}  R²={r2:.3f}"
    ).add_to(m)

colormap.add_to(m)
m.save('../results/geostatistics/spatial_interp_results_map.html')

In [9]:
# Vergleichskarte: alle drei Methoden auf separaten Tabs
from folium.plugins import GroupedLayerControl

methods = ['idw', 'ok', 'rk']

m2 = folium.Map(location=[51.5, 10.5], zoom_start=6, tiles='CartoDB positron')

for method in methods:
    df_m = results[results['method'] == method].copy()
    df_m = df_m.merge(coords[['park_id', 'latitude', 'longitude', 'altitude']],
                      left_on='station_id', right_on='park_id', how='left')

    median_r2 = df_m['r2'].median()
    layer = folium.FeatureGroup(name=f"{method.upper()} (median R²={median_r2:.3f})",
                                show=(method == 'ok'))

    for _, row in df_m.iterrows():
        r2 = row['r2']
        color = colormap(max(0, min(1, r2)))
        popup_html = (
            f"<b>Station {row['station_id']}</b><br>"
            f"R² = {r2:.3f}<br>"
            f"RMSE = {row['rmse']:.3f}<br>"
            f"MAE = {row['mae']:.3f}"
        )
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=7,
            color='black', weight=0.5,
            fill=True, fill_color=color, fill_opacity=0.85,
            popup=folium.Popup(popup_html, max_width=200),
            tooltip=f"{row['station_id']}  {method.upper()} R²={r2:.3f}"
        ).add_to(layer)

    layer.add_to(m2)

colormap.add_to(m2)
folium.LayerControl(collapsed=False).add_to(m2)
m2.save('../results/geostatistics/spatial_interp_results_comparison_map.html')